# LeetCode 3392 — Count Subarrays of Length Three With a Condition bằng Python

Link đề: https://leetcode.com/problems/count-subarrays-of-length-three-with-a-condition/ · Độ khó: **Easy** · Chủ đề: Array, Sliding Window (cửa sổ cố định), Math

Skeleton LeetCode:

```python
class Solution:
    def countSubarrays(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Biweekly Contest 146); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật. Trong notebook ta viết `list[int]` thay cho `List[int]` để không cần `import`; khi nộp LeetCode cả hai đều chạy được.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mảng `nums`, đếm các **mảng con (subarray) độ dài đúng 3** mà **tổng số đầu và số cuối bằng đúng một nửa số ở giữa**.
2. Nắm các công cụ Python: **chỉ số và lát cắt** `nums[i:i+3]`, `range(1, n - 1)` để duyệt **phần tử ở giữa**, `zip(nums, nums[1:], nums[2:])` để lấy **mọi cửa sổ độ dài 3**, `sum` trên giá trị `bool`, và sự khác nhau giữa `/`, `//` và phép nhân `* 2`.
3. Đi từ cách **cắt từng cửa sổ rồi kiểm tra** → **duyệt theo chỉ số phần tử giữa** → **`zip` ba lát cắt** → **ba biến trượt `a, b, c`** → **so sánh bằng phép nhân thay vì phép chia**.
4. Hiểu mấu chốt: điều kiện `a + c == b / 2` tương đương `(a + c) * 2 == b` — **chỉ dùng số nguyên**, không bị lỗi làm tròn của `//` (đặc biệt khi `b` lẻ hoặc **âm**).
5. Tránh các lỗi kinh điển: dùng `b // 2` (sai khi `b` lẻ), vòng lặp sai biên (`range(len(nums))` gây `IndexError`, hoặc bỏ sót cửa sổ cuối), đếm cửa sổ chồng lấp sai, và quên rằng mảng có thể chứa **số âm** và **số 0**.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Hãy trả về **số lượng mảng con có độ dài 3** sao cho **tổng của số thứ nhất và số thứ ba bằng đúng một nửa số thứ hai**.

Mảng con (subarray) là một đoạn **liên tiếp** các phần tử của mảng. Các mảng con có thể **chồng lên nhau** (ví dụ `[1,2,1]` và `[2,1,4]` chung hai phần tử), và mỗi vị trí bắt đầu được tính là **một** mảng con riêng.

**Ví dụ 1**

- Input: `nums = [1,2,1,4,1]`
- Output: `1`
- Giải thích: chỉ mảng con `[1,4,1]` thỏa: `1 + 1 = 2` và một nửa của `4` là `2`.

**Ví dụ 2**

- Input: `nums = [1,1,1]`
- Output: `0`
- Giải thích: mảng con duy nhất là `[1,1,1]`, nhưng `1 + 1 = 2` khác một nửa của `1` (là `0.5`).

**Ràng buộc:** `3 <= nums.length <= 100`; `-100 <= nums[i] <= 100`.

In [1]:
import random
from fractions import Fraction

examples = [
    ([1, 2, 1, 4, 1], 1),
    ([1, 1, 1], 0),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 1, 4, 1] -> kỳ vọng 1
[1, 1, 1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Chỉ số và lát cắt (slicing): `nums[i:i+3]`

Chỉ số của list bắt đầu từ `0`. Lát cắt `nums[i:j]` lấy các phần tử từ chỉ số `i` đến `j - 1` (**không gồm `j`**). Vì vậy `nums[i:i+3]` lấy **ba phần tử liên tiếp** bắt đầu từ `i`. Nếu cắt vượt quá cuối mảng, Python **không báo lỗi** mà chỉ trả về phần còn lại (có thể ngắn hơn 3 phần tử) — đây là điều cần cẩn thận.

In [2]:
nums = [10, 20, 30, 40, 50]
print(nums[0:3])        # [10, 20, 30]
print(nums[1:4])        # [20, 30, 40]
print(nums[2:5])        # [30, 40, 50]
print(nums[3:6])        # [40, 50]  <- chỉ có 2 phần tử, KHÔNG báo lỗi
print(len(nums[3:6]))   # 2

[10, 20, 30]
[20, 30, 40]
[30, 40, 50]
[40, 50]
2


### 3.2. `range(1, n - 1)` và phần tử ở giữa

Mỗi mảng con độ dài 3 có **một phần tử ở giữa**. Nếu mảng có `n` phần tử thì phần tử giữa có thể ở chỉ số `1, 2, ..., n - 2` (không thể là `0` vì không có phần tử bên trái, cũng không thể là `n - 1` vì không có phần tử bên phải). Đó chính là `range(1, n - 1)`. Số cửa sổ độ dài 3 là `n - 2`.

In [3]:
nums = [1, 2, 1, 4, 1]
n = len(nums)
print("n =", n, "-> số cửa sổ =", n - 2)
for j in range(1, n - 1):
    print(f"giữa ở chỉ số {j}: ({nums[j-1]}, {nums[j]}, {nums[j+1]})")

n = 5 -> số cửa sổ = 3
giữa ở chỉ số 1: (1, 2, 1)
giữa ở chỉ số 2: (2, 1, 4)
giữa ở chỉ số 3: (1, 4, 1)


### 3.3. `zip` nhiều dãy: ba lát cắt lệch nhau

`zip(a, b, c)` ghép các phần tử cùng vị trí của ba dãy và **dừng ở dãy ngắn nhất**. Nếu ta lấy `nums`, `nums[1:]`, `nums[2:]` (mỗi dãy lệch nhau một bước) thì bộ ba thứ `k` chính là `(nums[k], nums[k+1], nums[k+2])` — đúng một cửa sổ độ dài 3, và `zip` tự dừng đúng lúc, không bao giờ vượt biên.

In [4]:
nums = [1, 2, 1, 4, 1]
for a, b, c in zip(nums, nums[1:], nums[2:]):
    print(a, b, c)
print(list(zip(nums, nums[1:], nums[2:])))

1 2 1
2 1 4
1 4 1
[(1, 2, 1), (2, 1, 4), (1, 4, 1)]


### 3.4. `sum` trên giá trị `bool`

Trong Python `True` bằng `1` và `False` bằng `0` khi cộng. Vì vậy `sum(điều_kiện for ...)` **đếm số lần điều kiện đúng**. Đây là cách viết gọn thay cho `count = 0; ...; count += 1`.

In [5]:
print(True + True + False)                  # 2
flags = [3 > 1, 2 > 5, 4 == 4]
print(flags, sum(flags))                    # [True, False, True] 2
print(sum(x % 2 == 0 for x in [1, 2, 3, 4]))   # 2 số chẵn

2
[True, False, True] 2
2


### 3.5. `/`, `//` và vì sao nên nhân đôi

- `b / 2` là **phép chia thực** (cho kết quả `float`): `3 / 2 = 1.5`.
- `b // 2` là **chia lấy phần nguyên làm tròn xuống**: `3 // 2 = 1`, và với số âm `-3 // 2 = -2` (tròn **xuống**, không phải về `0`).

Đề nói tổng hai số **bằng đúng một nửa** số giữa. Nếu `b` lẻ thì một nửa của `b` là số có `.5`, mà tổng hai số nguyên không bao giờ có `.5`, nên mọi cửa sổ có `b` lẻ đều **không thỏa**. Dùng `//` sẽ vô tình "làm tròn" và cho kết quả sai. Cách an toàn: nhân hai vế với `2`: `a + c == b / 2` ⇔ `(a + c) * 2 == b`. Chỉ toàn số nguyên, không sai số.

In [6]:
print(3 / 2, 3 // 2)        # 1.5 1
print(-3 / 2, -3 // 2)      # -1.5 -2
a, b, c = 0, 3, 1           # a + c = 1; một nửa của b là 1.5 -> KHÔNG thỏa
print("dùng //  :", a + c == b // 2)      # True (SAI)
print("dùng * 2 :", (a + c) * 2 == b)       # False (ĐÚNG)

1.5 1
-1.5 -2
dùng //  : True
dùng * 2 : False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [1,2,1,4,1]` có `5 - 2 = 3` cửa sổ độ dài 3. Ta viết từng cửa sổ và kiểm tra: `(số đầu + số cuối) * 2` có bằng `số giữa` không?

| cửa sổ | `a` | `b` | `c` | `(a + c) * 2` | so với `b` | thỏa? |
|---|---|---|---|---|---|---|
| `[1,2,1]` | 1 | 2 | 1 | 4 | `4 != 2` | không |
| `[2,1,4]` | 2 | 1 | 4 | 12 | `12 != 1` | không |
| `[1,4,1]` | 1 | 4 | 1 | 4 | `4 == 4` | **có** |

Đáp án `1`. Bài này **không cần thuật toán cầu kỳ**: chỉ cần **duyệt từng cửa sổ độ dài 3 đúng một lần** và kiểm tra điều kiện. Câu hỏi thật sự là: viết vòng lặp **không sai biên** và **không sai phép so sánh** như thế nào.

In [7]:
nums = [1, 2, 1, 4, 1]
for i in range(len(nums) - 2):
    a, b, c = nums[i], nums[i + 1], nums[i + 2]
    print(f"[{a},{b},{c}]  (a+c)*2 = {(a + c) * 2}, b = {b}, thỏa = {(a + c) * 2 == b}")

[1,2,1]  (a+c)*2 = 4, b = 2, thỏa = False
[2,1,4]  (a+c)*2 = 12, b = 1, thỏa = False
[1,4,1]  (a+c)*2 = 4, b = 4, thỏa = True


## 5. Cách 1 — Brute Force / baseline: cắt từng cửa sổ rồi kiểm tra

**Ý tưởng.** Với mỗi vị trí bắt đầu `i` từ `0` đến `n - 3`, cắt `window = nums[i:i+3]`, lấy `window[0]`, `window[1]`, `window[2]` rồi kiểm tra điều kiện. Đây là cách "dịch thẳng đề bài sang code" — dễ đọc nhất, và đủ nhanh vì `n <= 100`.

In [8]:
def count_subarrays_slice(nums: list[int]) -> int:
    count = 0
    for i in range(len(nums) - 2):          # i = vị trí bắt đầu của cửa sổ
        window = nums[i:i + 3]              # 3 phần tử liên tiếp
        if (window[0] + window[2]) * 2 == window[1]:
            count += 1
    return count


for nums, expected in examples:
    print(count_subarrays_slice(nums), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Có `n - 2` cửa sổ, mỗi cửa sổ cắt `3` phần tử (`O(1)`) ⇒ **Time `O(n)`**. Mỗi lần cắt tạo list nhỏ mới nhưng bị bỏ ngay ⇒ **Space `O(1)`** phụ (không tính input).

**Nút thắt:** hầu như không có — thuật toán đã tối ưu về mặt bậc. Cái ta cải thiện thêm là **độ gọn** và **độ an toàn biên**: việc tạo list `window` mỗi vòng lặp là dư thừa, ta có thể đọc thẳng `nums[i]`, `nums[i+1]`, `nums[i+2]`.

## 6. Ý tưởng thuật toán / bước đột phá

Ba điều giúp code **gọn và khó sai**:

1. **Đổi phép chia thành phép nhân:** `a + c == b / 2` ⇔ `(a + c) * 2 == b`. Không `float`, không `//`.
2. **Neo vào phần tử giữa:** mỗi cửa sổ ứng với đúng **một** phần tử ở giữa `nums[j]` với `j` từ `1` đến `n - 2`. Khi đó hai hàng xóm là `nums[j-1]` và `nums[j+1]` — không cần lát cắt.
3. **Để `zip` lo phần biên:** `zip(nums, nums[1:], nums[2:])` tự dừng khi dãy ngắn nhất hết, nên không thể `IndexError`.

Tất cả chỉ là cùng một vòng lặp qua `n - 2` cửa sổ; khác nhau ở cách **đọc ba phần tử** của mỗi cửa sổ.

In [9]:
nums = [1, 4, 1, 0, 0, 0]
n = len(nums)
for j in range(1, n - 1):
    left, mid, right = nums[j - 1], nums[j], nums[j + 1]
    print(f"j={j}: left={left}, mid={mid}, right={right}, (left+right)*2={(left + right) * 2}, thỏa={(left + right) * 2 == mid}")

j=1: left=1, mid=4, right=1, (left+right)*2=4, thỏa=True
j=2: left=4, mid=1, right=0, (left+right)*2=8, thỏa=False
j=3: left=1, mid=0, right=0, (left+right)*2=2, thỏa=False
j=4: left=0, mid=0, right=0, (left+right)*2=0, thỏa=True


## 7. Các cách tiếp theo

### Cách 2 — Duyệt theo chỉ số phần tử giữa

**Giải thích từng đoạn.**

1. `for j in range(1, len(nums) - 1)`: `j` là chỉ số phần tử giữa, hai hàng xóm luôn tồn tại.
2. `(nums[j-1] + nums[j+1]) * 2 == nums[j]`: điều kiện đề bài, viết bằng số nguyên.
3. Mỗi lần thỏa, tăng `count`.

In [10]:
def count_subarrays_middle(nums: list[int]) -> int:
    count = 0
    for j in range(1, len(nums) - 1):
        if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]:
            count += 1
    return count


for nums, expected in examples:
    print(count_subarrays_middle(nums), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionMiddle:
    def countSubarrays(self, nums: list[int]) -> int:
        count = 0
        for j in range(1, len(nums) - 1):
            if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]:
                count += 1
        return count


print(SolutionMiddle().countSubarrays([1, 2, 1, 4, 1]))      # 1

1


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — `zip` ba lát cắt + `sum` (một dòng)

**Giải thích.** `zip(nums, nums[1:], nums[2:])` sinh mọi bộ ba `(a, b, c)` liên tiếp. Biểu thức `(a + c) * 2 == b` cho `True/False`; `sum` cộng chúng lại. Lưu ý `nums[1:]` và `nums[2:]` tạo thêm hai list sao chép nên tốn bộ nhớ `O(n)`.

In [12]:
def count_subarrays_zip(nums: list[int]) -> int:
    return sum((a + c) * 2 == b for a, b, c in zip(nums, nums[1:], nums[2:]))


for nums, expected in examples:
    print(count_subarrays_zip(nums), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionZip:
    def countSubarrays(self, nums: list[int]) -> int:
        return sum((a + c) * 2 == b for a, b, c in zip(nums, nums[1:], nums[2:]))


print(SolutionZip().countSubarrays([1, 2, 1, 4, 1]))         # 1

1


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (hai lát cắt `nums[1:]`, `nums[2:]` là bản sao). Với `n <= 100` hoàn toàn không đáng kể.

### Cách 4 — Cửa sổ trượt bằng ba biến `a, b, c`

**Giải thích.** Thay vì đọc lại `nums[...]` ba lần, ta giữ **ba biến** là ba phần tử của cửa sổ hiện tại. Mỗi bước trượt sang phải: `a ← b`, `b ← c`, `c ←` phần tử mới. Cách nghĩ này là mẫu của **sliding window cố định** và mở rộng được cho cửa sổ độ dài `k` bất kỳ.

In [14]:
def count_subarrays_rolling(nums: list[int]) -> int:
    a, b = nums[0], nums[1]                 # hai phần tử đầu của cửa sổ
    count = 0
    for c in nums[2:]:                      # c: phần tử mới vào cửa sổ
        if (a + c) * 2 == b:
            count += 1
        a, b = b, c                         # trượt cửa sổ sang phải một bước
    return count


for nums, expected in examples:
    print(count_subarrays_rolling(nums), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionRolling:
    def countSubarrays(self, nums: list[int]) -> int:
        a, b = nums[0], nums[1]
        count = 0
        for c in nums[2:]:
            if (a + c) * 2 == b:
                count += 1
            a, b = b, c
        return count


print(SolutionRolling().countSubarrays([1, 2, 1, 4, 1]))     # 1

1


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** nếu dùng chỉ số thay cho `nums[2:]` (với `nums[2:]` là `O(n)` nhưng nhỏ). Đề đảm bảo `n >= 3` nên `nums[0]`, `nums[1]` luôn tồn tại.

### Cách 5 — Dùng `Fraction` / `float` (chỉ để so sánh)

**Giải thích.** Nếu muốn bám sát câu chữ "bằng một nửa", ta có thể so `a + c` với `b / 2`. Với số nguyên nhỏ như đề (`|b| <= 100`) `float` biểu diễn chính xác `x.5`, nên `a + c == b / 2` vẫn đúng. Dùng `Fraction(b, 2)` còn **chính xác tuyệt đối** với mọi số nguyên. Cách này **không cần thiết** khi đã biết mẹo nhân đôi, nhưng giúp ta xác nhận mẹo là đúng.

In [16]:
def count_subarrays_fraction(nums: list[int]) -> int:
    count = 0
    for j in range(1, len(nums) - 1):
        if nums[j - 1] + nums[j + 1] == Fraction(nums[j], 2):
            count += 1
    return count


def count_subarrays_float(nums: list[int]) -> int:
    return sum(a + c == b / 2 for a, b, c in zip(nums, nums[1:], nums[2:]))


for nums, expected in examples:
    print(count_subarrays_fraction(nums), count_subarrays_float(nums), "(kỳ vọng", expected, ")")

1 1 (kỳ vọng 1 )
0 0 (kỳ vọng 0 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** (`Fraction` chậm hơn số nguyên một hằng số, không ảnh hưởng bậc).

## 8. TRACE từng bước

Ta trace Cách 2 trên `nums = [1,4,1,0,0,0,-1,-4,-1]` (có cả số `0` và số âm). Bảng trạng thái (trước khi quyết định ở mỗi bước):

| `j` | `nums[j-1]` | `nums[j]` | `nums[j+1]` | `(trái+phải)*2` | `== giữa`? | `count` sau bước |
|---|---|---|---|---|---|---|
| 1 | 1 | 4 | 1 | 4 | có | 1 |
| 2 | 4 | 1 | 0 | 8 | không | 1 |
| 3 | 1 | 0 | 0 | 2 | không | 1 |
| 4 | 0 | 0 | 0 | 0 | **có** (`0 == 0`) | 2 |
| 5 | 0 | 0 | -1 | -2 | không | 2 |
| 6 | 0 | -1 | -4 | -8 | không | 2 |
| 7 | -1 | -4 | -1 | -4 | **có** (`-4 == -4`) | 3 |

Hãy chú ý hai nhánh đặc biệt: `(0,0,0)` thỏa vì `0 + 0 = 0 = 0/2`; và `(-1,-4,-1)` thỏa với số âm.

In [17]:
def trace_count_subarrays(nums):
    count = 0
    print("nums =", nums, "| số cửa sổ =", len(nums) - 2)
    for j in range(1, len(nums) - 1):
        left, mid, right = nums[j - 1], nums[j], nums[j + 1]
        lhs = (left + right) * 2
        ok = lhs == mid
        print(f"j={j}: ({left}, {mid}, {right}) | (left+right)*2 = {lhs} | mid = {mid} | thỏa = {ok}")
        if ok:
            count += 1
            print(f"      -> count = {count}")
    print("kết quả:", count)
    return count


trace_count_subarrays([1, 4, 1, 0, 0, 0, -1, -4, -1])

nums = [1, 4, 1, 0, 0, 0, -1, -4, -1] | số cửa sổ = 7
j=1: (1, 4, 1) | (left+right)*2 = 4 | mid = 4 | thỏa = True
      -> count = 1
j=2: (4, 1, 0) | (left+right)*2 = 8 | mid = 1 | thỏa = False
j=3: (1, 0, 0) | (left+right)*2 = 2 | mid = 0 | thỏa = False
j=4: (0, 0, 0) | (left+right)*2 = 0 | mid = 0 | thỏa = True
      -> count = 2
j=5: (0, 0, -1) | (left+right)*2 = -2 | mid = 0 | thỏa = False
j=6: (0, -1, -4) | (left+right)*2 = -8 | mid = -1 | thỏa = False
j=7: (-1, -4, -1) | (left+right)*2 = -4 | mid = -4 | thỏa = True
      -> count = 3
kết quả: 3


3

## 9. Code nộp LeetCode

Ta chọn **Cách 2** (duyệt theo phần tử giữa): ngắn, đúng, dễ giải thích, `O(n)` thời gian và `O(1)` bộ nhớ.

In [18]:
class Solution:
    def countSubarrays(self, nums: list[int]) -> int:
        count = 0
        for j in range(1, len(nums) - 1):          # j: phần tử ở giữa của cửa sổ
            if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]:
                count += 1
        return count


for nums, expected in examples:
    print(Solution().countSubarrays(nums), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


## 10. Vì sao đúng

- **Tập cửa sổ đầy đủ, không trùng.** Mỗi mảng con độ dài 3 được xác định **duy nhất** bởi chỉ số phần tử giữa `j`; mọi `j` hợp lệ là `1..n-2`, và `range(1, n - 1)` đi qua **đúng** các giá trị đó, mỗi giá trị một lần. Vậy ta xét đúng `n - 2` cửa sổ, không sót, không lặp.
- **Điều kiện tương đương.** Với số nguyên: `a + c = b / 2` ⇔ `2(a + c) = b` (nhân hai vế với `2 > 0`, không đổi chiều, không đổi nghiệm). Phép nhân số nguyên là chính xác nên không có sai số làm tròn.
- **Số lẻ tự loại.** Nếu `b` lẻ, `2(a + c)` luôn chẵn nên không thể bằng `b`: đúng với nhận xét "tổng hai số nguyên không thể bằng `x.5`".
- **Số âm và số 0 vẫn đúng.** Phép nhân và so sánh bằng hoạt động đúng với số âm; `(0,0,0)` thỏa vì `0 = 0`.
- **Độ lệch biên.** Khi `n = 3` có `range(1, 2)` = `{1}`: đúng một cửa sổ.

## 11. Time/Space Complexity

Gọi `n = len(nums)`.

- **Time `O(n)`**: có `n - 2` cửa sổ, mỗi cửa sổ kiểm tra trong `O(1)`. Với `n <= 100` là tối đa `98` lần kiểm tra.
- **Space `O(1)`**: chỉ dùng biến `count` và `j`.

Không thể tốt hơn `O(n)` về bậc, vì mỗi phần tử phải được đọc ít nhất một lần (bỏ qua một phần tử có thể làm đổi đáp án).

## 12. Bộ test

### 12.1. Ví dụ của đề và gom các cách vào một bảng

In [19]:
impls = {
    "slice": count_subarrays_slice,
    "middle": count_subarrays_middle,
    "zip": count_subarrays_zip,
    "rolling": count_subarrays_rolling,
    "fraction": count_subarrays_fraction,
    "float": count_subarrays_float,
    "Solution": lambda nums: Solution().countSubarrays(nums),
}

for nums, expected in examples:
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums)
print("PASS các ví dụ của đề với", len(impls), "cách")

PASS các ví dụ của đề với 7 cách


### 12.2. Edge case

- `n = 3` (đúng một cửa sổ): thỏa và không thỏa.
- Toàn số `0`: mọi cửa sổ đều thỏa.
- Số âm, số lẻ ở giữa, `b = 0` nhưng `a + c != 0`.
- Giá trị biên `-100` và `100`.

Mỗi kỳ vọng dưới đây đều được tính tay rồi kiểm lại bằng một cách độc lập (xem 12.4 với `Fraction`). Ví dụ `[50, 100, 50]` **không** thỏa vì `(50 + 50) * 2 = 200 != 100`; còn `[25, 100, 25]` thỏa vì `(25 + 25) * 2 = 100`.

In [20]:
cases = [
    ([0, 0, 0], 1),
    ([1, 2, 1], 0),
    ([1, 4, 1], 1),
    ([0, 0, 0, 0, 0], 3),
    ([-1, -4, -1], 1),
    ([-1, -3, -1], 0),
    ([1, 0, 1], 0),
    ([50, 100, 50], 0),                 # (50+50)*2 = 200 != 100
    ([25, 100, 25], 1),
    ([-50, -100, -50], 0),
    ([-25, -100, -25], 1),
    ([100, -100, 100], 0),
    ([100, 100, -100], 0),
    ([1, 3, 0, 3, 1, 2, 4], 0),
]
for nums, expected in cases:
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums, fn(nums), expected)
print("PASS", len(cases), "edge case x", len(impls), "cách")

PASS 14 edge case x 7 cách


### 12.3. Phản ví dụ: vì sao `//` sai

Ta chủ động dựng các cửa sổ khiến `a + c == b // 2` nhưng **không** thỏa `a + c == b / 2`.

In [21]:
def wrong_floor_div(nums):
    return sum(a + c == b // 2 for a, b, c in zip(nums, nums[1:], nums[2:]))


bad_inputs = [[0, 3, 1], [1, 5, 1], [-1, -3, -1], [2, 7, 1]]
for nums in bad_inputs:
    print(nums, "| sai (//):", wrong_floor_div(nums), "| đúng:", Solution().countSubarrays(nums))
    assert wrong_floor_div(nums) != Solution().countSubarrays(nums)

[0, 3, 1] | sai (//): 1 | đúng: 0
[1, 5, 1] | sai (//): 1 | đúng: 0
[-1, -3, -1] | sai (//): 1 | đúng: 0
[2, 7, 1] | sai (//): 1 | đúng: 0


### 12.4. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Ta sinh `n` từ `3` đến `100`, giá trị từ `-100` đến `100`. Vì cửa sổ thỏa rất hiếm với dữ liệu ngẫu nhiên đều, ta **chủ động chèn** các bộ ba thỏa (`b = 2(a + c)`) để có nhiều trường hợp dương tính.

In [22]:
rng = random.Random(3392)
positives = 0
for trial in range(3000):
    n = rng.randint(3, 100)
    nums = [rng.randint(-100, 100) for _ in range(n)]
    if trial % 2 == 0:                               # chèn một vài bộ ba thỏa
        for _ in range(rng.randint(1, 5)):
            j = rng.randint(1, n - 2)
            a, c = rng.randint(-25, 25), rng.randint(-25, 25)
            nums[j - 1], nums[j], nums[j + 1] = a, 2 * (a + c), c
    assert 3 <= len(nums) <= 100 and all(-100 <= x <= 100 for x in nums)
    ref = count_subarrays_fraction(nums)             # cách độc lập, chính xác
    positives += ref > 0
    for name, fn in impls.items():
        assert fn(nums) == ref, (name, nums)
print("PASS 3000 mảng ngẫu nhiên; số mảng có đáp án > 0:", positives)

PASS 3000 mảng ngẫu nhiên; số mảng có đáp án > 0: 1650


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `b // 2`

In [23]:
def sai_chia_nguyen(nums):
    count = 0
    for j in range(1, len(nums) - 1):
        if nums[j - 1] + nums[j + 1] == nums[j] // 2:      # SAI: // làm tròn xuống
            count += 1
    return count


print("sai:", sai_chia_nguyen([0, 3, 1]), "| đúng:", Solution().countSubarrays([0, 3, 1]))
print("sai:", sai_chia_nguyen([-1, -3, -1]), "| đúng:", Solution().countSubarrays([-1, -3, -1]))

sai: 1 | đúng: 0
sai: 1 | đúng: 0


**Vì sao sai:** `3 // 2 = 1` nên `0 + 1 == 1` được tính là thỏa, trong khi một nửa của `3` là `1.5`. Với số âm `-3 // 2 = -2` (tròn xuống) nên `-1 + -1 == -2` cũng bị tính nhầm. **Cách sửa:** nhân đôi tổng `(a + c) * 2 == b`.

### Lỗi 2: `range` sai biên → `IndexError`

In [24]:
def sai_range_thieu_bien(nums):
    count = 0
    for j in range(len(nums)):                       # SAI: j chạy tới n - 1, nums[j + 1] vượt biên
        if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]:
            count += 1
    return count


try:
    sai_range_thieu_bien([1, 4, 1])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** khi `j = n - 1` thì `nums[j + 1]` không tồn tại; khi `j = 0` thì `nums[j - 1] = nums[-1]` là **phần tử cuối** (Python cho chỉ số âm quay vòng) — một cửa sổ "quấn vòng" không có thật. **Cách sửa:** `range(1, len(nums) - 1)`.

### Lỗi 3: Bỏ sót cửa sổ cuối hoặc đếm thừa

In [25]:
def sai_thieu_cua_so_cuoi(nums):
    count = 0
    for j in range(1, len(nums) - 2):                # SAI: dừng sớm một bước
        if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]:
            count += 1
    return count


print("sai:", sai_thieu_cua_so_cuoi([1, 2, 1, 4, 1]), "| đúng:", Solution().countSubarrays([1, 2, 1, 4, 1]))
print("sai:", sai_thieu_cua_so_cuoi([1, 4, 1]), "| đúng:", Solution().countSubarrays([1, 4, 1]))

sai: 0 | đúng: 1
sai: 0 | đúng: 1


**Vì sao sai:** với `n = 5` đáng ra `j ∈ {1, 2, 3}`, nhưng `range(1, 3)` chỉ cho `{1, 2}` nên bỏ mất cửa sổ cuối `[1,4,1]`. **Cách sửa:** điểm dừng của `range` là `n - 1` (không gồm), tức `range(1, n - 1)`. Hãy luôn thử `n = 3` (đúng một cửa sổ).

### Lỗi 4: Đọc nhầm đề — so với phần tử đầu hoặc cuối thay vì giữa

In [26]:
def sai_doc_de(nums):
    return sum(a + b == c / 2 for a, b, c in zip(nums, nums[1:], nums[2:]))   # SAI: lấy hai số đầu, một nửa số cuối


print("sai:", sai_doc_de([1, 2, 1, 4, 1]), "| đúng:", Solution().countSubarrays([1, 2, 1, 4, 1]))
print("sai:", sai_doc_de([1, 1, 4]), "| đúng:", Solution().countSubarrays([1, 1, 4]))

sai: 0 | đúng: 1
sai: 1 | đúng: 0


**Vì sao sai:** đề nói tổng **số thứ nhất và số thứ ba** bằng nửa **số thứ hai**. Hãy gán tên `a, b, c` rồi viết điều kiện `(a + c) * 2 == b` thay vì trộn lẫn. **Cách sửa:** dùng tên biến rõ nghĩa (`left`, `mid`, `right`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Mảng con độ dài cố định `k`**" ⇒ **sliding window cố định**; với `k = 3` thường viết thẳng `nums[i-1], nums[i], nums[i+1]` hoặc `zip(nums, nums[1:], nums[2:])`.
- Số cửa sổ độ dài `k` của mảng `n` phần tử là `n - k + 1` (`n - 2` khi `k = 3`). Dùng nó để kiểm tra biên.
- Điều kiện có **chia** (nửa, trung bình, tỉ lệ) ⇒ **nhân chéo** để chỉ dùng số nguyên.
- Cẩn thận `//` với số **âm**: nó làm tròn **xuống** chứ không về `0`.
- Chỉ số âm trong Python **quay vòng**: `nums[-1]` là phần tử cuối — nguồn của nhiều lỗi biên âm thầm (không báo lỗi!).
- Luôn thử `n` nhỏ nhất của ràng buộc (`n = 3`) và dữ liệu toàn `0`.
- Khi kỳ vọng tính tay, kiểm lại bằng một cách độc lập (như `Fraction`).

## 15. Biến thể của bài

1. **Độ dài cửa sổ `k` tổng quát:** đếm cửa sổ độ dài `k` thỏa điều kiện nào đó (tổng, trung bình...).
2. **Trả về danh sách** các chỉ số bắt đầu của cửa sổ thỏa thay vì số lượng.
3. **Điều kiện khác:** `a + c == b` (không có "một nửa"), hoặc `a < b < c` (cửa sổ tăng dần), hoặc `b` là **trung bình** của `a` và `c` (cấp số cộng độ dài 3).
4. **Chuỗi ký tự thay vì số**, ví dụ đếm cửa sổ độ dài 3 có ba ký tự đôi một khác nhau (LeetCode 1876).
5. Bài liên quan: **Count Good Triplets (1534)**, **Arithmetic Slices (413)**, **Number of Sub-arrays of Size K and Average Greater than or Equal to Threshold (1343)**, **Substrings of Size Three with Distinct Characters (1876)**.

In [27]:
def window_k_count(nums, k, predicate):
    """Đếm cửa sổ độ dài k thỏa predicate(window)."""
    return sum(predicate(nums[i:i + k]) for i in range(len(nums) - k + 1))


def start_indices(nums):
    return [j - 1 for j in range(1, len(nums) - 1) if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]]


def count_arithmetic_triples(nums):
    """b là trung bình của a và c: 2b == a + c."""
    return sum(2 * b == a + c for a, b, c in zip(nums, nums[1:], nums[2:]))


print(window_k_count([1, 2, 1, 4, 1], 3, lambda w: (w[0] + w[2]) * 2 == w[1]))   # 1
print(start_indices([1, 4, 1, 0, 0, 0, -1, -4, -1]))                            # [0, 3, 6]
print(count_arithmetic_triples([1, 2, 3, 4, 6]))                                # 2 -> (1,2,3), (2,3,4)
assert window_k_count([1, 2, 1, 4, 1], 3, lambda w: (w[0] + w[2]) * 2 == w[1]) == 1
assert start_indices([1, 4, 1, 0, 0, 0, -1, -4, -1]) == [0, 3, 6]
assert count_arithmetic_triples([1, 2, 3, 4, 6]) == 2

1
[0, 3, 6]
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Cắt `nums[i:i+3]` | tạo list cửa sổ rồi kiểm tra | `O(n)` | `O(1)` | dễ đọc nhất, hợp người mới |
| 2. Chỉ số phần tử giữa | `range(1, n-1)`, đọc hàng xóm | `O(n)` | `O(1)` | **nên nộp** |
| 3. `zip` ba lát cắt | `sum(... for a, b, c in zip(...))` | `O(n)` | `O(n)` | một dòng, gọn |
| 4. Ba biến trượt | `a, b = b, c` | `O(n)` | `O(1)` | mẫu sliding window tổng quát |
| 5. `Fraction` / `float` | so với `b / 2` | `O(n)` | `O(1)` | đối chiếu, không khuyến khích khi nộp |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(zip([1,2,3,4], [2,3,4], [3,4]))`, `[1,2,3,4,5][3:6]`, `7 // 2`, `-7 // 2`, `Solution().countSubarrays([2,8,2,0,0,0])`.

**Bài 2 (tính tay).** Với `nums = [3, 6, 0, 2, 4, 0, 0]`: liệt kê từng cửa sổ độ dài 3 và `(a + c) * 2`, rồi đếm số cửa sổ thỏa.

**Bài 3 (điền TODO).** Hoàn thành `count_subarrays_todo(nums)` theo Cách 2.

In [28]:
def count_subarrays_todo(nums):
    # TODO 1: khởi tạo count = 0
    # TODO 2: duyệt j từ 1 đến len(nums) - 2 (dùng range(1, len(nums) - 1))
    # TODO 3: nếu (nums[j - 1] + nums[j + 1]) * 2 == nums[j] thì tăng count
    # TODO 4: trả về count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `count_subarrays_k3_zip(nums)` bằng **một biểu thức `sum`** và `zip` như Cách 3, nhưng thay lát cắt `nums[1:]`, `nums[2:]` (tạo bản sao) bằng `itertools.islice(nums, 1, None)` và `itertools.islice(nums, 2, None)` (không sao chép). Giải thích khác biệt về bộ nhớ.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_arithmetic_slices3(nums)` đếm cửa sổ độ dài 3 là **cấp số cộng** (`b - a == c - b`). (b) Viết `count_windows(nums, k)` đếm số cửa sổ độ dài `k` có **tổng bằng `0`** (dùng tổng tiền tố hoặc cộng/trừ khi trượt).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[(1,2,3), (2,3,4)]`, `[4, 5]`, `3`, `-4` (tròn xuống), `2` (hai cửa sổ thỏa: `(2,8,2)` vì `4*2 = 8`, và `(0,0,0)` vì `0 = 0`). Bài 3: `count = 0; for j in range(1, len(nums) - 1): ...`.
- **Mức 2 (vừa):** Bài 2: các cửa sổ `(3,6,0)`: `6 == 6` ✔; `(6,0,2)`: `16 != 0`; `(0,2,4)`: `8 != 2`; `(2,4,0)`: `(2+0)*2 = 4 == 4` ✔; `(4,0,0)`: `8 != 0`. Có `2` cửa sổ thỏa. Bài 4: `zip` dừng ở dãy ngắn nhất nên không sợ vượt biên.
- **Mức 3 (sâu):** Bài 5(a): `sum(b - a == c - b for a, b, c in zip(nums, nums[1:], nums[2:]))`. 5(b): tổng cửa sổ đầu `s = sum(nums[:k])`, rồi mỗi bước `s += nums[i + k] - nums[i]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert list(zip([1, 2, 3, 4], [2, 3, 4], [3, 4])) == [(1, 2, 3), (2, 3, 4)]
assert [1, 2, 3, 4, 5][3:6] == [4, 5]
assert 7 // 2 == 3
assert -7 // 2 == -4
assert Solution().countSubarrays([2, 8, 2, 0, 0, 0]) == 2     # (2,8,2) và (0,0,0)

# Bài 2: (3,6,0) thỏa, (2,4,0) thỏa
nums2 = [3, 6, 0, 2, 4, 0, 0]
assert Solution().countSubarrays(nums2) == 2
assert [(a, b, c) for a, b, c in zip(nums2, nums2[1:], nums2[2:]) if (a + c) * 2 == b] == [(3, 6, 0), (2, 4, 0)]


# Bài 3
def count_subarrays_todo_ans(nums):
    count = 0
    for j in range(1, len(nums) - 1):
        if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]:
            count += 1
    return count


for nums, expected in examples:
    assert count_subarrays_todo_ans(nums) == expected


# Bài 4
from itertools import islice


def count_subarrays_k3_zip(nums):
    return sum((a + c) * 2 == b for a, b, c in zip(nums, islice(nums, 1, None), islice(nums, 2, None)))


for nums, expected in examples:
    assert count_subarrays_k3_zip(nums) == expected
assert count_subarrays_k3_zip([1, 4, 1, 0, 0, 0, -1, -4, -1]) == 3


# Bài 5
def count_arithmetic_slices3(nums):
    return sum(b - a == c - b for a, b, c in zip(nums, nums[1:], nums[2:]))


def count_windows(nums, k):
    s = sum(nums[:k])
    count = 1 if s == 0 else 0
    for i in range(len(nums) - k):
        s += nums[i + k] - nums[i]
        count += s == 0
    return count


assert count_arithmetic_slices3([1, 2, 3, 4, 6]) == 2
assert count_windows([1, -1, 2, -2, 0], 2) == 2          # (1,-1) và (2,-2); (-1,2) = 1 và (-2,0) = -2 không thỏa
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm cửa sổ độ dài 3 theo phần tử giữa, so sánh bằng phép nhân)

```python
class Solution:
    def countSubarrays(self, nums: List[int]) -> int:
        count = 0
        for j in range(1, len(nums) - 1):
            if (nums[j - 1] + nums[j + 1]) * 2 == nums[j]:
                count += 1
        return count
```

**Checklist**

- [ ] So sánh `(a + c) * 2 == b` thay vì `//` hoặc `/`?
- [ ] `range(1, len(nums) - 1)`: không `IndexError`, không bỏ sót cửa sổ cuối?
- [ ] Đã thử `n = 3` và mảng toàn `0` (mọi cửa sổ thỏa)?
- [ ] Đã thử số âm và `b` lẻ (không bao giờ thỏa)?
- [ ] Skeleton đúng `class Solution` / `def countSubarrays(self, nums: List[int]) -> int`?
- [ ] Độ phức tạp `O(n)` thời gian, `O(1)` bộ nhớ?